In [1]:
!pip install transformers huggingface_hub datasets wandb evaluate rouge_score accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00


In [2]:
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    TrainingArguments,
    Trainer
)
from datasets import load_dataset
import numpy as np
import evaluate
import torch
import wandb
import os

In [3]:
import os
os.environ["WANDB_DISABLED"] = "true"
print("Skipped login — not needed for training")

Skipped login — not needed for training


In [4]:
import os
os.environ["WANDB_DISABLED"] = "true"
print("Ready")

Ready


In [5]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/sheeetalthapa/medical-jargons/simplification_data.json


In [6]:
data_files = '/kaggle/input/datasets/sheeetalthapa/medical-jargons/simplification_data.json'  
dataset = load_dataset("json", data_files=data_files)
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['medical', 'simple'],
        num_rows: 6967
    })
})


In [7]:
from datasets import DatasetDict

# Dataset loaded as dict with 'train' key already
print(dataset)
print(type(dataset))

# Access correctly
if 'train' in dataset:
    full = dataset['train']
else:
    # It's already a flat dataset
    full = dataset

num_samples = len(full)
num_train = int(0.8 * num_samples)
num_val   = int(0.1 * num_samples)
num_test  = num_samples - num_train - num_val

shuffled = full.shuffle(seed=42)
train_dataset = shuffled.select(range(num_train))
val_dataset   = shuffled.select(range(num_train, num_train + num_val))
test_dataset  = shuffled.select(range(num_train + num_val, num_samples))

dataset_dict = DatasetDict({
    'train': train_dataset,
    'valid': val_dataset,
    'test':  test_dataset
})
print(dataset_dict)
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

DatasetDict({
    train: Dataset({
        features: ['medical', 'simple'],
        num_rows: 6967
    })
})
<class 'datasets.dataset_dict.DatasetDict'>
DatasetDict({
    train: Dataset({
        features: ['medical', 'simple'],
        num_rows: 5573
    })
    valid: Dataset({
        features: ['medical', 'simple'],
        num_rows: 696
    })
    test: Dataset({
        features: ['medical', 'simple'],
        num_rows: 698
    })
})
Train: 5573 | Val: 696 | Test: 698


In [8]:
model_name = "google-t5/t5-base"
new_model = "t5-base-ft-medical-simplifier"

tokenizer = T5Tokenizer.from_pretrained(model_name)

model = T5ForConditionalGeneration.from_pretrained(model_name)

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [9]:
def preprocess_function(examples):
    inputs = [f"simplify: {text}" for text in examples['medical']]
    targets = [text for text in examples['simple']]
    
    model_inputs = tokenizer(
        inputs, 
        max_length=512, 
        truncation=True, 
        padding='max_length'
    )
    
    labels = tokenizer(
        text_target=targets,  # ← fixed: use text_target instead of as_target_tokenizer
        max_length=512, 
        truncation=True, 
        padding='max_length'
    )
    
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

print("preprocess_function defined")

preprocess_function defined


In [10]:
tokenized_dataset = dataset_dict.map(
    preprocess_function,
    batched=True,
)

Map:   0%|          | 0/5573 [00:00<?, ? examples/s]

Map:   0%|          | 0/696 [00:00<?, ? examples/s]

Map:   0%|          | 0/698 [00:00<?, ? examples/s]

In [11]:
rouge = evaluate.load('rouge')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    # Handle different prediction formats
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    
    # Clip to valid token range
    predictions = np.clip(predictions, 0, tokenizer.vocab_size - 1).astype(np.int32)
    
    decoded_preds  = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = np.clip(labels, 0, tokenizer.vocab_size - 1).astype(np.int32)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]
    
    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
        rouge_types=['rouge1','rouge2','rougeL']
    )
    
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) 
                       for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)
    
    return {k: round(v, 4) for k, v in result.items()}

print("compute_metrics defined ✅")

compute_metrics defined ✅


In [12]:
def preprocess_logits_for_metrics(logits, labels):
    pred_ids = torch.argmax(logits[0], dim=-1)
    return pred_ids, labels

In [13]:
device = torch.device("cuda") if torch.cuda.is_available() else "cpu"
model.to(device)

os.environ["WANDB_DISABLED"] = "true"

training_args = TrainingArguments(
    output_dir='medical_simplifer_t5_base_results',
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=1000,
    weight_decay=0.01,
    logging_dir="logs",
    logging_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=5,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    learning_rate=3e-5,
)
print("Training args ready ")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training args ready 


In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['valid'],
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    compute_metrics=compute_metrics
)

trainer.train()

wandb.finish()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Gen Len
500,8.601168,0.102339,0.686900,0.470700,0.680000,23.808900
1000,0.110496,0.078184,0.750000,0.551600,0.743500,23.849100
1500,0.088435,0.067301,0.777100,0.595300,0.771100,23.849100
2000,0.078458,0.061687,0.791700,0.619500,0.786800,23.849100
2500,0.069295,0.058010,0.802500,0.641200,0.798600,23.849100
3000,0.066303,0.055193,0.810400,0.650900,0.806500,23.849100
3500,0.061793,0.053370,0.817900,0.663000,0.815000,23.849100
4000,0.058490,0.051575,0.819800,0.666700,0.816200,23.849100
4500,0.055067,0.050433,0.826100,0.680100,0.822700,23.849100
5000,0.054672,0.049570,0.829400,0.685500,0.826200,23.849100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


In [15]:
trainer.model.save_pretrained(new_model)
model.config.use_cache=True
model.eval()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

In [16]:
def simplify(text, model, tokenizer, max_length=512, num_beams=2):
    
    inputs = tokenizer.encode(
        text,
        return_tensors='pt',
        max_length=max_length,
        truncation=True
    ).to(device)
    
    generated_ids = model.generate(
        inputs,
        max_new_tokens=1024,
        num_beams=num_beams,
        early_stopping=True
    )
    
    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)

In [17]:
trainer.model.push_to_hub(new_model, use_temp_dir=False)

TypeError: PushToHubMixin.push_to_hub() got an unexpected keyword argument 'use_temp_dir'

In [ ]:
trainer.push_to_hub(new_model)

In [ ]:
tokenizer.push_to_hub(new_model)